# Semana 01 · Tracking MLOps con MLflow en Databricks

En esta práctica vas a entrenar modelos **muy sencillos** para centrarte en el ciclo MLOps: datos, parámetros, métricas, artefactos, comparación y una decisión reproducible. El objetivo no es ganar una competición de machine learning, sino dejar evidencia clara en MLflow.


## Mapa y criterios de terminación

Al acabar debes poder enseñar:

1. un experimento de MLflow con varios runs comparables;
2. tags que identifiquen tu alias, lote y propósito docente;
3. parámetros y métricas de validación/test;
4. un modelo ganador guardado como artefacto;
5. un pequeño smoke test de inferencia.


## 0. Dependencias

En Databricks, ejecuta `%pip` al principio. Si el entorno reinicia Python, continúa desde la celda de imports.


In [0]:
%pip install -q --upgrade "mlflow[databricks]" "scikit-learn" "pandas" "matplotlib"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Configuración e identidad del lote

Usa rutas de Databricks (`/dbfs/...`) para artefactos pequeños que quieras inspeccionar fuera de MLflow.


In [0]:
import json
import time
import uuid
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
STUDENT_ALIAS = "alejandroldz2004@gmail.com"  # TODO 1: decide un alias estable; importa porque lo usarás para filtrar tus runs; inspecciona tags de MLflow; verifica que aparece en search_runs.
BATCH_ID = f"s01-{uuid.uuid4().hex[:8]}"
EXPERIMENT_NAME = "/Shared/muiaap-operacion-modelos/semana01-tracking"
ARTIFACT_DIR = Path("/tmp/muiaap_s01_tracking")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_experiment(EXPERIMENT_NAME)
print({"batch_id": BATCH_ID, "alias": STUDENT_ALIAS, "artifact_dir": str(ARTIFACT_DIR)})

{'batch_id': 's01-006899d0', 'alias': 'alejandroldz2004@gmail.com', 'artifact_dir': '/tmp/muiaap_s01_tracking'}


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## 2. Datos pequeños y contrato mínimo

Usaremos `load_wine` de scikit-learn: no descarga datos, es rápido y suficiente para practicar tracking.


In [0]:
wine = load_wine(as_frame=True)
data = wine.frame.rename(columns={"target": "quality_class"})
FEATURES = list(wine.feature_names)
TARGET = "quality_class"

assert TARGET in data.columns
assert data[FEATURES].isna().sum().sum() == 0
assert data[TARGET].nunique() == 3

display(data.head())
print(data.shape, data[TARGET].value_counts())

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,quality_class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


(178, 14) quality_class
1    71
0    59
2    48
Name: count, dtype: int64


## 3. Separar train/validación/test

La celda ya trae una partición sencilla para que la práctica sea ejecutable en Databricks.

**TODO 2:** revisa si el reparto 70/15/15 mantiene todas las clases en validación y test. Decide si cambiarías el porcentaje de test en un proyecto real, porque la evaluación debe ser comparable y suficientemente estable. Inspecciona `train_test_split` y el parámetro `stratify`. Verifica con los `assert` y los conteos por clase.


In [0]:
X = data[FEATURES]
y = data[TARGET]

# Partición base 70/15/15. TODO 2: revisa los tamaños y conteos de clase antes de aceptar este split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

assert len(X_train) + len(X_valid) + len(X_test) == len(data)
print({"train": len(X_train), "valid": len(X_valid), "test": len(X_test)})
print("valid classes:", y_valid.value_counts().sort_index().to_dict())
print("test classes:", y_test.value_counts().sort_index().to_dict())

# Es correcto. Los conjuntos corresponden al 70/15/15 % que se menciona. Ademas los conjuntos valid y train mantienen las proporciones del conjunto de datos original gracias al stratify


{'train': 124, 'valid': 27, 'test': 27}
valid classes: {0: 9, 1: 10, 2: 8}
test classes: {0: 9, 1: 11, 2: 7}


## 4. Candidatos simples

Mantén pocos candidatos para que el foco esté en MLflow. Un baseline, una regresión logística y un bosque pequeño son suficientes.

**TODO 3:** revisa los parámetros y decide si cambiarías uno. Importa porque los parámetros deben explicar diferencias entre runs. Inspecciona la API del estimador elegido y verifica que cada candidato tiene nombre y modelo.


In [0]:
#añadiria el random_state a los parámetros, ya que si este cambia (y por tanto los resultados), no lo estamos teniendo en cuenta

CANDIDATES = [
    {
        "name": "baseline_most_frequent",
        "model": DummyClassifier(strategy="most_frequent"),
        "params": {"family": "baseline", "strategy": "most_frequent"},
    },
    {
        "name": "logistic_regression",
        "model": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
        ]),
        "params": {"family": "linear", "max_iter": 500, "random_state" : SEED}, #random_state incluido
    },
    {
        "name": "random_forest_small",
        "model": RandomForestClassifier(n_estimators=60, max_depth=4, random_state=SEED),
        "params": {"family": "trees", "n_estimators": 60, "max_depth": 4, "random_state" : SEED}, #random_state incluido
    },
]

assert all("name" in c and "model" in c for c in CANDIDATES)
[c["name"] for c in CANDIDATES]

['baseline_most_frequent', 'logistic_regression', 'random_forest_small']

## 5. Registrar runs comparables

Cada run debe contar la misma historia: identidad, parámetros, métrica de validación y artefactos mínimos.

**TODO 4:** completa o revisa los tags. Importa porque después filtrarás sólo tus runs. Inspecciona `mlflow.set_tags`, `mlflow.log_params`, `mlflow.log_metrics` y `mlflow.sklearn.log_model`. Verifica que `run_infos` contiene un `model_uri` por candidato.


In [0]:
run_infos = []

for candidate in CANDIDATES:
    with mlflow.start_run(run_name=f"{BATCH_ID}-{candidate['name']}") as run:
        mlflow.set_tags({
            "course": "operacion-modelos",
            "week": "01",
            "student.alias": STUDENT_ALIAS,
            "batch_id": BATCH_ID,
            "purpose": "guided-mlflow-tracking",
        })
        mlflow.log_params(candidate["params"])
        mlflow.log_param("feature_count", len(FEATURES))
        mlflow.log_param("train_rows", len(X_train))
        mlflow.log_param("validation_rows", len(X_valid))

        start = time.perf_counter()
        model = candidate["model"]
        model.fit(X_train, y_train)
        valid_pred = model.predict(X_valid)
        latency_ms = (time.perf_counter() - start) * 1000

        metrics = {
            "validation.accuracy": accuracy_score(y_valid, valid_pred),
            "validation.f1_macro": f1_score(y_valid, valid_pred, average="macro"),
            "train_latency_ms": latency_ms,
        }
        mlflow.log_metrics(metrics)

        report = classification_report(y_valid, valid_pred, output_dict=True)
        report_path = ARTIFACT_DIR / f"{candidate['name']}_validation_report.json"
        report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(report_path), artifact_path="validation")
        model_info = mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(3),     skops_trusted_types=["sklearn.tree._tree.Tree"],) #añado skops_trusted_types por que me daba fallo

        run_infos.append({
            "run_id": run.info.run_id,
            "name": candidate["name"],
            "model_uri": model_info.model_uri,
            **metrics,
        })

runs_df = pd.DataFrame(run_infos).sort_values("validation.f1_macro", ascending=False)
display(runs_df)
assert len(runs_df) == len(CANDIDATES)
assert runs_df["model_uri"].notna().all()

2026/10/02 17:26:50 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o11.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

,run_id,name,model_uri,validation.accuracy,validation.f1_macro,train_latency_ms
2,44547dbf06c9441aa33b11be1b3a615f,random_forest_small,models:/m-bba4e15cfaa84ccfa8b84cb291438be7,1.000000,1.000000,84.428021
1,aa30b672091b465fb827e8cb7acd3f65,logistic_regression,models:/m-db7e9fc598544e07bb736f1d6c62528f,0.962963,0.964912,97.581363
0,2eed1504ffe14d4fb36cf30d9a4925b7,baseline_most_frequent,models:/m-5de5e35fee304083a064072a04a88c12,0.370370,0.180180,5.229442


## 6. Buscar tus runs y elegir ganador sin mirar test

**TODO 5:** filtra por `student.alias` y `batch_id`. Importa porque en un workspace compartido hay runs de otras personas. Inspecciona `mlflow.search_runs` y verifica que no aparecen runs de otro lote.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
filter_string = f"tags.student.alias = '{STUDENT_ALIAS}' and tags.batch_id = '{BATCH_ID}'"
searched = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=filter_string,
    order_by=["metrics.validation.f1_macro DESC"],
)

display(searched[["run_id", "tags.mlflow.runName", "metrics.validation.f1_macro", "metrics.validation.accuracy"]])
assert set(searched["tags.batch_id"]) == {BATCH_ID}

best_row = searched.iloc[0]
BEST_RUN_ID = best_row["run_id"]
BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"
print({"best_run_id": BEST_RUN_ID, "best_model_uri": BEST_MODEL_URI})

,run_id,tags.mlflow.runName,metrics.validation.f1_macro,metrics.validation.accuracy
0,44547dbf06c9441aa33b11be1b3a615f,s01-006899d0-random_forest_small,1.000000,1.000000
1,aa30b672091b465fb827e8cb7acd3f65,s01-006899d0-logistic_regression,0.964912,0.962963
2,2eed1504ffe14d4fb36cf30d9a4925b7,s01-006899d0-baseline_most_frequent,0.180180,0.370370


{'best_run_id': '44547dbf06c9441aa33b11be1b3a615f', 'best_model_uri': 'runs:/44547dbf06c9441aa33b11be1b3a615f/model'}


## 7. Evaluar test una sola vez

Test se usa después de elegir. Esto evita optimizar contra la evidencia final.

**TODO 6:** registra métricas de test en el run ganador. Importa porque la decisión queda trazada en el mismo run. Inspecciona `mlflow.start_run(run_id=...)` y verifica que las métricas `test.*` aparecen en MLflow.


In [0]:
best_model = mlflow.sklearn.load_model(BEST_MODEL_URI)
test_pred = best_model.predict(X_test)
test_metrics = {
    "test.accuracy": accuracy_score(y_test, test_pred),
    "test.f1_macro": f1_score(y_test, test_pred, average="macro"),
}

with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tags({"selection.stage": "tested", "selection.reason": "best_validation_f1_macro"})

print(test_metrics)

{'test.accuracy': 1.0, 'test.f1_macro': 1.0}


## 8. Smoke test de inferencia

Comprueba que un consumidor puede cargar el modelo por URI y predecir pocas filas. No levantamos API local ni usamos CLI: eso llegará en semanas posteriores.


In [0]:
sample = X_test.head(3)
predictions = best_model.predict(sample)
result = pd.DataFrame({"prediction": predictions}, index=sample.index)
display(pd.concat([sample.reset_index(drop=True), result.reset_index(drop=True)], axis=1))
assert len(predictions) == 3
assert set(predictions).issubset(set(y.unique()))

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,prediction
0,14.06,2.15,2.61,17.6,121.0,2.60,2.51,0.31,1.25,5.05,1.06,3.58,1295.0,0
1,14.38,3.59,2.28,16.0,102.0,3.25,3.17,0.27,2.19,4.90,1.04,3.44,1065.0,0
2,12.93,3.80,2.65,18.6,102.0,2.41,2.41,0.25,1.98,4.50,1.03,3.52,770.0,0


## Entregable

Incluye en tu ficha: `EXPERIMENT_NAME`, `BATCH_ID`, `BEST_RUN_ID`, la métrica de validación usada para elegir, la métrica de test y una captura del smoke test.


In [0]:
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("BATCH_ID:", BATCH_ID)
print("STUDENT_ALIAS:", STUDENT_ALIAS)
print("BEST_RUN_ID:", BEST_RUN_ID)
print("BEST_MODEL_URI:", BEST_MODEL_URI)

print("Métrica de validación usada para elegir")
print("validation.f1_macro (best run):", runs_df.loc[runs_df['run_id'] == BEST_RUN_ID, 'validation.f1_macro'].values[0])
print("validation.accuracy (best run):", runs_df.loc[runs_df['run_id'] == BEST_RUN_ID, 'validation.accuracy'].values[0])

print("Métricas de test (una sola vez)")
print("test.accuracy:", test_metrics["test.accuracy"])
print("test.f1_macro:", test_metrics["test.f1_macro"])

print("Smoke test de inferencia (3 filas)")
pd.concat([sample.reset_index(drop=True), result.reset_index(drop=True)], axis=1)


EXPERIMENT_NAME: /Shared/muiaap-operacion-modelos/semana01-tracking
BATCH_ID: s01-006899d0
STUDENT_ALIAS: alejandroldz2004@gmail.com
BEST_RUN_ID: 44547dbf06c9441aa33b11be1b3a615f
BEST_MODEL_URI: runs:/44547dbf06c9441aa33b11be1b3a615f/model
Métrica de validación usada para elegir
validation.f1_macro (best run): 1.0
validation.accuracy (best run): 1.0
Métricas de test (una sola vez)
test.accuracy: 1.0
test.f1_macro: 1.0
Smoke test de inferencia (3 filas)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,prediction
0,14.06,2.15,2.61,17.6,121.0,2.60,2.51,0.31,1.25,5.05,1.06,3.58,1295.0,0
1,14.38,3.59,2.28,16.0,102.0,3.25,3.17,0.27,2.19,4.90,1.04,3.44,1065.0,0
2,12.93,3.80,2.65,18.6,102.0,2.41,2.41,0.25,1.98,4.50,1.03,3.52,770.0,0


![image_1790962664351.png](./image_1790962664351.png "image_1790962664351.png")